In [1]:
!pip install -q transformers accelerate bitsandbytes fastapi uvicorn pyngrok nest-asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00


**MODEL**

In [ ]:
import sys
import traceback
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    import torch

    model_id = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto"
    )

    print("torch.cuda.is_available() =", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("torch.cuda.get_device_name(0) =", torch.cuda.get_device_name(0))
    print("next(model.parameters()).device =", next(model.parameters()).device)
    print("model.hf_device_map =", getattr(model, "hf_device_map", None))
    if torch.cuda.is_available():
        print("torch.cuda.memory_allocated() =", torch.cuda.memory_allocated())
        print("torch.cuda.memory_reserved() =", torch.cuda.memory_reserved())
    with open("status.txt", "w") as f:
        f.write("OK")
except Exception as e:
    with open("status.txt", "w") as f:
        f.write(traceback.format_exc())


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

In [ ]:
from google.colab import output
import requests
import time
import json
try:
    res = requests.post("http://localhost:8000/plan", json={"text": "create a file test.txt"}, timeout=60)
    data = {
        "status": res.status_code,
        "json": res.json()
    }
    output.eval_js(f'window.top.postMessage("TEST_RESULT:" + {repr(json.dumps(data))}, "*")')
except Exception as e:
    output.eval_js(f'window.top.postMessage("TEST_RESULT:ERROR:" + {repr(str(e))}, "*")')


In [ ]:
from fastapi import FastAPI
import uvicorn
import nest_asyncio
import threading
import time
import torch

app = FastAPI()

@app.get("/health")
def health():
    return {
        "status": "ok",
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
        "model": "Qwen/Qwen2.5-Coder-1.5B-Instruct",
        "loaded": True
    }

@app.post("/plan")
def plan(data: dict):
    t0 = time.perf_counter()
    def log_stage(stage: str, last_time: float):
        now = time.perf_counter()
        elapsed = now - last_time
        print(f"[PLAN] {stage} ({elapsed * 1000 if elapsed < 1 else elapsed:.2f} {'ms' if elapsed < 1 else 's'})")
        return now

    print(f"[PLAN] Request received")
    t_curr = t0

    text = data["text"]
    
    t_curr = log_stage("Before preprocessing", t_curr)
    messages = [
        {"role": "system", "content": "You are a JSON assistant. Respond only with a single valid JSON object containing the plan. Do not include markdown code block formatting (like ```json), explanations, comments, or any extra text."},
        {"role": "user", "content": text}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    t_curr = log_stage("Tokenizer start", t_curr)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    t_curr = log_stage("Tokenizer end", t_curr)

    t_curr = log_stage("Generate start", t_curr)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=False
    )
    t_curr = log_stage("Generate end", t_curr)

    t_curr = log_stage("Decode start", t_curr)
    generated_ids = outputs[0][inputs.input_ids.shape[-1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    t_curr = log_stage("Decode end", t_curr)

    t_curr = log_stage("Before returning response", t_curr)
    
    if response.startswith("```"):
        lines = response.splitlines()
        if lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]
        response = "\n".join(lines).strip()

    total = time.perf_counter() - t0
    print(f"[PLAN] Total request time: {total:.2f} s")
    
    return {"response": response}

nest_asyncio.apply()

threading.Thread(
    target=lambda: uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    ),
    daemon=True
).start()


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3FXnKeOXI9lIfnM1iSifaoXz1W8_5KYtzmvdit3cdx1uSEpxu")

public_url = ngrok.connect(8000)

print(public_url)

In [ ]:
from google.colab import output
import requests
import json
try:
    headers = {"ngrok-skip-browser-warning": "true"}
    res = requests.post("https://evaluator-agreeing-plenty.ngrok-free.dev/plan", json={"text": "write a python hello world"}, headers=headers, timeout=60)
    data = {
        "status": res.status_code,
        "json": res.json()
    }
    output.eval_js(f'window.top.postMessage("TEST_RESULT:" + {repr(json.dumps(data))}, "*")')
except Exception as e:
    output.eval_js(f'window.top.postMessage("TEST_RESULT:ERROR:" + {repr(str(e))}, "*")')
